# Pseudo-Differential Solvers, Propagators & Ray Dynamics

A guided tour of the `psiop` solver layer, illustrating **non-trivial** examples of:

**Engines**
- `characteristic_hamiltonians` — branch decomposition (scalar & matrix symbols)
- `integrate_singularity` — bicharacteristic / ray integration (incl. chaotic Hénon–Heiles)
- `build_propagator` — asymptotic exponential propagators, *validated against exact multipliers and `scipy.linalg.expm`*
- `solve_first_order` / `solve_second_order` — IVP solvers (variable-coefficient advection–diffusion, Dirac wave-packet splitting, Klein–Gordon scattering on a potential barrier)
- `solve_matrix_field` / `solve_sylvester_field` — matrix-field evolutions (Sylvester validated against the exact Fourier-space formula)
- `solve_ricci_flow_conformal_2d` — geometric flow with invariant checks (area, Gauss–Bonnet)

**Graphics**
- `plot_scalar_1d`, `plot_matrix_1d`, `plot_scalar_2d`, `animate_scalar_1d`
- `plot_matrix_field_1d`, `plot_matrix_field_2d`, `plot_wave_solution_1d`
- `animate_singularity`, `animate_singularity_3d`

**Convention.** Evolution equations are written `∂ₜu = Op(s)u`. For a physical Hamiltonian `H` (Schrödinger `i∂ₜu = Hu`) the generator symbol is `s = -i·H`, and `characteristic_hamiltonians` returns the branches `H_k = Re(i·λ_k)` of the symbol matrix.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.linalg import expm
from IPython.display import HTML, display

from psiop import (
    characteristic_hamiltonians, integrate_singularity, build_propagator,
    solve_first_order, solve_second_order,
    solve_matrix_field, solve_sylvester_field, solve_ricci_flow_conformal_2d,
    plot_scalar_1d, plot_matrix_1d, plot_scalar_2d, animate_scalar_1d,
    plot_matrix_field_1d, plot_matrix_field_2d, plot_wave_solution_1d,
    animate_singularity, animate_singularity_3d,
    make_grid_1d, PseudoDifferentialOperator,
)

plt.rcParams['figure.dpi'] = 100
print('psiop solver layer loaded.')

## Part 1 — Characteristic Hamiltonians & Ray Dynamics

`characteristic_hamiltonians(s, vars_x)` extracts the ray Hamiltonians `H_k = Re(i·λ_k)`:
- **scalar** symbol → one branch;
- **matrix** symbol → one branch per eigenvalue of the symbol matrix (e.g. the ± energy sheets of a Dirac operator).

`integrate_singularity` then integrates Hamilton's equations `ẋ = ∂H/∂ξ`, `ξ̇ = −∂H/∂x` from an initial phase-space point.

Three increasingly rich examples:
1. **Harmonic oscillator** — closed circular orbits; energy must be conserved to solver tolerance (sanity check).
2. **Massive 1D Dirac operator** — a single incoming singularity *splits* into two rays with opposite group velocities.
3. **2D Hénon–Heiles** — chaotic ray + Poincaré section.

In [ ]:
x, xi = sp.symbols('x xi', real=True)
y, eta = sp.symbols('y eta', real=True)

# --- Scalar generator: Schrödinger equation with the harmonic oscillator ---
#   i d_t u = H u,  H = (xi^2 + x^2)/2   ==>   s = -i H
s_ho = -sp.I * (xi**2 + x**2) / 2
H_ho, _, _ = characteristic_hamiltonians(s_ho, [x])
print('Scalar generator -> 1 characteristic branch:')
sp.pprint(H_ho[0])

# --- Matrix generator: 1D Dirac operator with mass m = 1 ---
#   i d_t u = (xi sigma_z + sigma_x) u
m_mass = 1
S_dirac = -sp.I * sp.Matrix([[xi, m_mass], [m_mass, -xi]])
H_dirac, _, _ = characteristic_hamiltonians(S_dirac, [x])
print('\nDirac generator -> 2 characteristic branches (+/- energy):')
for Hk in H_dirac:
    sp.pprint(sp.simplify(Hk))

In [ ]:
# Harmonic oscillator ray: circular orbit on the energy shell
x0_r, xi0_r = 1.0, 0.0
_, _, _, t_ray, trajs = integrate_singularity(
    s_ho, [x], x0=x0_r, xi0=xi0_r, tmax=4*np.pi, n_frames=400,
    method='DOP853', rtol=1e-10, atol=1e-12)
xr, xir = trajs[0]
E_ray = 0.5*(xr**2 + xir**2)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
theta = np.linspace(0, 2*np.pi, 200)
R = np.hypot(x0_r, xi0_r)
ax[0].plot(xr, xir, lw=1.5, label='ray')
ax[0].plot(R*np.cos(theta), R*np.sin(theta), 'k--', lw=0.8, label='energy shell')
ax[0].set_xlabel('x'); ax[0].set_ylabel(r'$\xi$')
ax[0].set_title(r'Bicharacteristic of $H=(x^2+\xi^2)/2$')
ax[0].legend(); ax[0].set_aspect('equal'); ax[0].grid(alpha=0.3)

ax[1].semilogy(t_ray, np.abs(E_ray - E_ray[0]) + 1e-16)
ax[1].set_xlabel('t'); ax[1].set_ylabel(r'$|H(t)-H(0)|$')
ax[1].set_title('Energy conservation along the ray')
ax[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

In [ ]:
# Dirac: two branches from the same initial point -> opposite group velocities
_, _, _, t_d, trajs_d = integrate_singularity(
    S_dirac, [x], x0=0.0, xi0=2.0, tmax=6.0, n_frames=150)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))

# Left: dispersion curves H_+- (xi) = +- sqrt(xi^2 + m^2)
xi_curve = np.linspace(-5, 5, 400)
ax[0].plot(xi_curve,  np.sqrt(xi_curve**2 + m_mass**2), 'tab:red',
           label=r'$H_+ = +\sqrt{\xi^2+m^2}$')
ax[0].plot(xi_curve, -np.sqrt(xi_curve**2 + m_mass**2), 'tab:blue',
           label=r'$H_- = -\sqrt{\xi^2+m^2}$')
ax[0].axvline(2.0, color='k', ls=':', lw=1)
ax[0].set_xlabel(r'$\xi$'); ax[0].set_ylabel(r'$H(\xi)$')
ax[0].set_title('Dirac branches: two energy sheets')
ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)

# Right: x(t) along each branch (dxi/dt = -dH/dx = 0, xi stays at xi0)
colors = ['tab:red', 'tab:blue']
for k, Yk in enumerate(trajs_d):
    xk = Yk[0]
    v = np.polyfit(t_d, xk, 1)[0]
    ax[1].plot(t_d, xk, color=colors[k % 2], lw=2,
               label=f'branch {k+1},  dx/dt = {v:+.3f}')
ax[1].set_xlabel('t'); ax[1].set_ylabel('x(t)')
ax[1].set_title(r'Singularity splitting at $\xi_0=2$: speeds $\pm 2/\sqrt{5}$')
ax[1].legend(fontsize=9); ax[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

In [ ]:
# Hénon–Heiles Hamiltonian: the classic chaotic system
V_hh = (x**2 + y**2)/2 + x**2*y - y**3/3
H_hh = (xi**2 + eta**2)/2 + V_hh
s_hh = -sp.I * H_hh

E_target = 0.125                       # below the escape threshold
xi0_hh, eta0_hh = 0.40, 0.30           # E = (0.4^2 + 0.3^2)/2 = 0.125
_, _, _, t_hhr, trajs_hh = integrate_singularity(
    s_hh, [x, y], x0=[0.0, 0.0], xi0=[xi0_hh, eta0_hh],
    tmax=400.0, n_frames=4000, method='DOP853', rtol=1e-10, atol=1e-12)

# trajs_hh is a list with one entry per characteristic branch;
# a scalar symbol has exactly one branch -> take trajs_hh[0]
Xr, Yr, XIr, ETAr = trajs_hh[0]
E_num = 0.5*(XIr**2 + ETAr**2) + 0.5*(Xr**2 + Yr**2) + Xr**2*Yr - Yr**3/3

# Poincaré section: upward crossings of y = 0
poin_x, poin_xi = [], []
for i in range(len(t_hhr) - 1):
    if Yr[i] <= 0 < Yr[i+1] and ETAr[i+1] > 0:
        frac = -Yr[i] / (Yr[i+1] - Yr[i])
        poin_x.append(Xr[i] + frac*(Xr[i+1] - Xr[i]))
        poin_xi.append(XIr[i] + frac*(XIr[i+1] - XIr[i]))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
ax[0].plot(Xr, Yr, lw=0.5, color='tab:blue')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
ax[0].set_title('Ray in configuration space (t <= 400)')
ax[0].set_aspect('equal'); ax[0].grid(alpha=0.3)

ax[1].plot(poin_x, poin_xi, '.', ms=3, color='tab:red')
ax[1].set_xlabel('x'); ax[1].set_ylabel(r'$\xi$')
ax[1].set_title(r'Poincaré section ($y=0$, $\eta>0$), $E=%.3f$' % E_target)
ax[1].grid(alpha=0.3)

ax[2].semilogy(t_hhr, np.abs(E_num - E_num[0]) + 1e-16, lw=0.7)
ax[2].set_xlabel('t'); ax[2].set_ylabel(r'$|H(t)-H(0)|$')
ax[2].set_title('Energy drift')
ax[2].grid(alpha=0.3)
fig.tight_layout(); plt.show()
print(f'Poincaré points collected: {len(poin_x)}')

### Animations of the ray flow

`animate_singularity` (2D projection) and `animate_singularity_3d` animate the bicharacteristics. In 1D we request `projection='phase'` to draw `(x, ξ)`; the 3D version draws the tube `(x, ξ, t)` in 1D and `(x, y, ξ)` in 2D.

*(Fixed: the axes are now sized to the actual trajectory range with a small padding margin. They previously stayed at matplotlib's default `(0, 1)` box regardless of the data, since the trail/point artists are created empty and only ever updated via `set_data`.)*

In [ ]:
# Harmonic oscillator: singularity rotating on the energy shell (phase projection)
anim_ho = animate_singularity(s_ho, [x], x0=1.0, xi0=0.0, tmax=2*np.pi,
                              n_frames=80, projection='phase', interval=40)
HTML(anim_ho.to_jshtml())

In [ ]:
# Dirac: one singularity splits into two rays (animate all branches)
anim_dirac = animate_singularity(S_dirac, [x], x0=0.0, xi0=2.0, tmax=6.0,
                                 n_frames=80, projection='phase',
                                 branches='all', interval=40)
HTML(anim_dirac.to_jshtml())

In [ ]:
# 3D tube (x, xi, t) for the oscillator
anim_ho_3d = animate_singularity_3d(s_ho, [x], x0=1.0, xi0=0.0,
                                    tmax=4*np.pi, n_frames=120, interval=40)
HTML(anim_ho_3d.to_jshtml())

In [ ]:
# Chaotic Hénon–Heiles ray in (x, y, xi)
anim_hh_3d = animate_singularity_3d(s_hh, [x, y], x0=[0.0, 0.0],
                                    xi0=[xi0_hh, eta0_hh], tmax=120.0,
                                    n_frames=250, interval=30)
HTML(anim_hh_3d.to_jshtml())

## Part 2 — `build_propagator`: asymptotic exponential propagators

`build_propagator(s, vars_x, dt, order)` builds an operator whose symbol approximates

`exp(dt·Op(s)) ≈ I + dt·P + (dt²/2!)·P∘P + (dt³/3!)·P∘P∘P + …`

(truncated asymptotic composition). For **constant-coefficient** symbols, composition reduces to exact multiplication, so the only error is the Taylor truncation in `dt` — which lets us validate against:
- the exact Fourier multiplier `exp(dt·s(ξ))` (scalar case),
- `scipy.linalg.expm(dt·S(ξ))` (matrix case).

In [ ]:
# Scalar propagator: convergence of the symbol and of one application step
dt_p = 0.10
s_gen = -xi**2 - 1.5*sp.I*xi          # d_t u = u_xx - 1.5 u_x  (heat + transport)
exact_mult = sp.lambdify(xi, sp.exp(dt_p*s_gen), 'numpy')
ks = np.linspace(-6, 6, 600)
p_exact = exact_mult(ks)

props, sym_err = {}, {}
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
for order in [1, 2, 3, 4]:
    prop, is_mat, size = build_propagator(s_gen, [x], dt=dt_p, order=order)
    props[order] = prop
    p_num = sp.lambdify(xi, prop.symbol, 'numpy')(ks)
    sym_err[order] = np.abs(p_num - p_exact)
    ax[0].semilogy(ks, sym_err[order] + 1e-16, label=f'order {order}')
ax[0].set_xlabel(r'$\xi$'); ax[0].set_ylabel(r'$|p_{prop} - e^{dt\,s}|$')
ax[0].set_title('Propagator symbol vs exact multiplier')
ax[0].legend(); ax[0].grid(alpha=0.3)

# One-step application on a Gaussian, compared with the exact multiplier
xg, kx = make_grid_1d(L=8.0, N=256)
u0 = np.exp(-xg**2)
u_exact = np.fft.ifft(exact_mult(kx) * np.fft.fft(u0))
app_err = [np.max(np.abs(props[o].apply(u0, xg, kx,
                                         freq_window=None, clamp=np.inf) - u_exact))
           for o in [1, 2, 3, 4]]
ax[1].semilogy([1, 2, 3, 4], app_err, 'o-', color='tab:red')
ax[1].set_xlabel('asymptotic order'); ax[1].set_ylabel(r'$\|u_{prop}-u_{exact}\|_\infty$')
ax[1].set_title('One-step application error (constant coefficients)')
ax[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

In [ ]:
# See the exact multiplier and each propagator's symbol directly (not just their error)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].plot(ks, np.abs(p_exact), 'k-', lw=2.5, label='exact')
for order in [1, 2, 3, 4]:
    p_num = sp.lambdify(xi, props[order].symbol, 'numpy')(ks)
    ax[0].plot(ks, np.abs(p_num), '--', label=f'order {order}')
ax[0].set_xlabel(r'$\xi$'); ax[0].set_ylabel(r'$|p(\xi)|$')
ax[0].set_title('Magnitude of the propagator symbol')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].plot(ks, np.angle(p_exact), 'k-', lw=2.5, label='exact')
for order in [1, 2, 3, 4]:
    p_num = sp.lambdify(xi, props[order].symbol, 'numpy')(ks)
    ax[1].plot(ks, np.angle(p_num), '--', label=f'order {order}')
ax[1].set_xlabel(r'$\xi$'); ax[1].set_ylabel(r'arg $p(\xi)$ [rad]')
ax[1].set_title('Phase of the propagator symbol')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

ax[2].plot(xg, u_exact.real, 'k-', lw=2.5, label='exact')
for order in [1, 2, 3, 4]:
    u_prop = props[order].apply(u0, xg, kx, freq_window=None, clamp=np.inf)
    ax[2].plot(xg, u_prop.real, '--', label=f'order {order}')
ax[2].set_xlabel('x'); ax[2].set_ylabel(r'Re $u(x, dt)$')
ax[2].set_title('One-step solution profile')
ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)

fig.tight_layout(); plt.show()


In [ ]:
# Matrix propagator: compare with scipy.linalg.expm at a sample frequency
dt_d = 0.05
prop_dirac, is_mat, size = build_propagator(S_dirac, [x], dt=dt_d, order=4)
print(f'is_matrix = {is_mat}, size = {size}')

xi_star = 1.3
S_num = np.asarray(sp.lambdify(xi, S_dirac, 'numpy')(xi_star), dtype=complex)
E_exact = expm(dt_d * S_num)
E_prop = prop_dirac.symbol_matrix(0.0, xi_star)

print('\nexp(dt*S(xi*)) via scipy.linalg.expm:')
print(np.round(E_exact, 6))
print('\nasymptotic propagator, symbol_matrix(0, xi*):')
print(np.round(E_prop, 6))
print(f'\nmax entrywise error = {np.max(np.abs(E_prop - E_exact)):.3e}')

## Part 3 — `solve_first_order`: `∂ₜu = Op(s)u`

Two evolutions of increasing richness:
1. **Scalar, variable coefficients** — advection–diffusion with a periodic speed profile `c(x)` (`plot_scalar_1d` + `animate_scalar_1d`).
2. **2×2 Dirac system** — a wave packet launched in one component splits into two packets travelling at opposite group velocities; the flow is unitary, so `‖u(t)‖²` must be conserved (`plot_matrix_1d`).

In [ ]:
# Variable-coefficient advection–diffusion on a periodic domain
L1 = 10.0
c_x = 0.5 + 0.3*sp.sin(sp.pi*x/L1)     # periodic speed profile
nu = 0.02
s_advdiff = -sp.I*c_x*xi - nu*xi**2

t_ad, U_ad, (xg_ad, kx_ad) = solve_first_order(
    s_advdiff, [x], lambda X: np.exp(-X**2),
    dt=0.01, n_steps=500, order=2, L=L1, N=256,
    apply_kwargs=dict(freq_window='gaussian'))

fig_ad = plot_scalar_1d(t_ad, U_ad, xg_ad,
                        title='variable-coefficient advection–diffusion',
                        quantity='real', n_snapshots=6)
display(fig_ad)

In [ ]:
# Same solution, animated
anim_ad = animate_scalar_1d(t_ad, U_ad, xg_ad, quantity='real', interval=40)
HTML(anim_ad.to_jshtml())

In [ ]:
# 2x2 Dirac system: wave-packet splitting + unitarity check
k0 = 3.0
f_vec = lambda X: [np.exp(-X**2) * np.exp(1j*k0*X),
                   np.zeros_like(X, dtype=complex)]
t_dir, U_dir, (xg_dir, kx_dir) = solve_first_order(
    S_dirac, [x], f_vec, dt=0.02, n_steps=150, order=4, L=10.0, N=256,
    apply_kwargs=dict(freq_window=None, clamp=np.inf))

fig_dir = plot_matrix_1d(t_dir, U_dir, xg_dir,
                         labels=[r'$u_1$', r'$u_2$'], quantity='abs')
display(fig_dir)

# Unitarity: the generator is self-adjoint -> ||u||^2 must be conserved
dx_dir = xg_dir[1] - xg_dir[0]
norm2 = dx_dir * np.sum(np.abs(U_dir)**2, axis=(1, 2))
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(t_dir, norm2/norm2[0] - 1.0)
ax.set_xlabel('t'); ax.set_ylabel(r'$\|u(t)\|^2/\|u_0\|^2 - 1$')
ax.set_title('L2-norm conservation (unitary Dirac flow)')
ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

## Part 4 — `solve_second_order`: `∂ₜₜu = Op(s)u`

Klein–Gordon-type scattering on a Gaussian potential barrier:

`∂ₜₜu = ∂ₓₓu − V(x)u`,  `V(x) = 4·e^{−x²/2}`.

The incoming packet (carrier `k₀ = 4`, launched at `x = −5` and moving right) is **partially transmitted and partially reflected** — visible simultaneously in `u` and `∂ₜu` via `plot_wave_solution_1d`.

In [ ]:
V_barrier = 4.0*sp.exp(-x**2/2)          # barrier at x = 0
s_wave = -xi**2 - V_barrier

x_c, sigma_w, k0_w = -5.0, 1.0, 4.0
def f_inc(X):     # incoming packet
    a = X - x_c
    return np.exp(-a**2/sigma_w**2) * np.cos(k0_w*a)
def g_inc(X):     # -d/dx f_inc  ->  right-moving packet
    a = X - x_c
    return np.exp(-a**2/sigma_w**2) * (
        2*a/sigma_w**2 * np.cos(k0_w*a) + k0_w*np.sin(k0_w*a))

t_w, U_w, V_w, (xg_w, kx_w) = solve_second_order(
    s_wave, [x], f_inc, g_inc, dt=0.02, n_steps=300, order=3, L=10.0, N=512,
    apply_kwargs=dict(freq_window='gaussian'))

fig_w = plot_wave_solution_1d(t_w, U_w, V_w, xg_w, quantity='real')
display(fig_w)

## Part 5 — 2D evolution: quantum Hénon–Heiles (`plot_scalar_2d`)

The same Hénon–Heiles Hamiltonian as Part 1 now drives a 2D wave packet, `∂ₜu = −i·H·u`. We then superimpose the classical bicharacteristic launched from the packet's center `(x₀, p₀)`: by Ehrenfest's theorem the quantum mass follows the (chaotic) ray for some time.

In [ ]:
def f_wavepacket(X, Y):
    gauss = np.exp(-((X - 0.1)**2 + (Y - 0.1)**2) / (2*0.5**2))
    phase = np.exp(1j*(0.45*X + 0.35*Y))
    return gauss * phase

t_q, U_q, grids_q = solve_first_order(
    s_hh, [x, y], f_wavepacket, dt=0.005, n_steps=200, order=2, L=6.0, N=96,
    apply_kwargs=dict(freq_window='gaussian'))
xg_q, yg_q = grids_q[0], grids_q[1]
n_q = len(t_q)

fig_q = plot_scalar_2d(t_q, U_q, xg_q, yg_q, quantity='abs',
                       times=[0, n_q//3, 2*n_q//3, n_q - 1])
display(fig_q)

In [ ]:
# Ehrenfest correspondence: classical ray over the final |u|
_, _, _, _, trajs_q = integrate_singularity(
    s_hh, [x, y], x0=[0.1, 0.1], xi0=[0.45, 0.35],
    tmax=t_q[-1], n_frames=400)
Xc, Yc = trajs_q[0][0], trajs_q[0][1]

fig, ax = plt.subplots(figsize=(5.8, 5.2))
ax.pcolormesh(xg_q, yg_q, np.abs(U_q[-1]).T, shading='auto', cmap='inferno')
ax.plot(Xc, Yc, '-', color='deepskyblue', lw=1.3, label='bicharacteristic')
ax.plot([Xc[0]], [Yc[0]], 'o', color='deepskyblue', label='launch point')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'|u(x, y)| at t = {t_q[-1]:.2f} with the classical ray')
ax.legend(); ax.set_aspect('equal')
fig.tight_layout(); plt.show()

## Part 6 — `solve_matrix_field`: N×N matrix fields

`∂ₜU = P·U` where `U(x)` is itself a matrix (density matrix / matrix Green's function) and `P` acts by left multiplication.

- **1D:** `P = diag(−iξ, 2iξ)` → row 1 advects at speed +1, row 2 at speed −2. Illustrated with `plot_matrix_field_1d` and `component='diag'`, `'trace'`, `'frobenius'`, `(i, j)`.
- **2D:** row 1 advects along `x`, row 2 along `y` → `plot_matrix_field_2d`.

In [ ]:
# 1D matrix field: rows advected at different speeds
S_mf = sp.Matrix([[-sp.I*xi, 0], [0, 2*sp.I*xi]])
def F_mf(X):
    return np.array([
        [np.exp(-X**2),             0.5*np.exp(-(X - 1.0)**2)],
        [0.5*np.exp(-(X + 1.0)**2), np.exp(-X**2)]
    ])

t_mf, U_mf, (xg_mf, kx_mf) = solve_matrix_field(
    S_mf, [x], F_mf, dt=0.05, n_steps=60, order=3, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian'))

fig_mf_diag = plot_matrix_field_1d(t_mf, U_mf, xg_mf, component='diag',
                                   quantity='real',
                                   labels=[r'$U_{11}$', r'$U_{22}$'])
display(fig_mf_diag)

In [ ]:
# Other reductions of the same matrix field
fig_tr = plot_matrix_field_1d(t_mf, U_mf, xg_mf, component='trace', quantity='real')
display(fig_tr)

fig_fr = plot_matrix_field_1d(t_mf, U_mf, xg_mf, component='frobenius')
display(fig_fr)

fig_01 = plot_matrix_field_1d(t_mf, U_mf, xg_mf, component=(0, 1), quantity='abs')
display(fig_01)

In [ ]:
# 2D matrix field: row 1 advected along x, row 2 along y
S_mf2 = sp.Matrix([[-sp.I*xi, 0], [0, -sp.I*eta]])
def F_mf2(X, Y):
    b1 = np.exp(-((X + 1.0)**2 + Y**2))
    b2 = np.exp(-(X**2 + (Y + 1.0)**2))
    return np.array([[b1, 0.5*b2], [0.5*b1, b2]])

t_m2, U_m2, grids_m2 = solve_matrix_field(
    S_mf2, [x, y], F_mf2, dt=0.05, n_steps=80, order=3, L=6.0, N=64,
    apply_kwargs=dict(freq_window='gaussian'))
xg_m2, yg_m2 = grids_m2[0], grids_m2[1]

fig_m2_fr = plot_matrix_field_2d(t_m2, U_m2, xg_m2, yg_m2,
                                 component='frobenius', times=[0, 40, 79])
display(fig_m2_fr)

fig_m2_00 = plot_matrix_field_2d(t_m2, U_m2, xg_m2, yg_m2,
                                 component=(0, 0), quantity='real',
                                 times=[0, 40, 79])
display(fig_m2_00)

## Part 7 — `solve_sylvester_field`: `∂ₜU = P·U − U·Q`

Left Dirac mixing (`P`) + right heat damping (`Q`), advanced with **Strang splitting**.

Because both symbols are constant-coefficient Fourier multipliers, the exact solution is known pointwise in Fourier space:

`Û(t, k) = exp(t·P(k)) · Û₀(k) · exp(−t·Q(k))`

— we use it to validate the splitting solver quantitatively.

In [ ]:
# d_t U = P U - U Q : left Dirac mixing + right heat damping
P_syl = sp.Matrix([[0, -sp.I*xi], [-sp.I*xi, 0]])
Q_syl = sp.Matrix([[xi**2, 0], [0, xi**2]])
def F_syl(X):
    g = np.exp(-X**2)
    return np.array([[g, 0.5*g], [0.5*g, g]])

# STABILITY FIX: the order-3 heat propagator is the Taylor polynomial
# T3(-z) = 1 - z + z^2/2 - z^3/6 with z = dt*k^2, stable only for z < ~2.5.
# With N=128, L=10 (kmax ~ 20, kmax^2 ~ 400), dt=0.02 gave z ~ 8 -> |T3| ~ 60,
# i.e. ~60x amplification of high-k round-off PER step (the 1e196 blow-up).
# dt = 0.004 gives z_max ~ 1.6: strictly contractive, no windowing needed.
dt_sy = 0.004
t_sy, U_sy, (xg_sy, kx_sy) = solve_sylvester_field(
    P_syl, Q_syl, [x], F_syl, dt=dt_sy, n_steps=300, order=3,
    L=10.0, N=128, splitting='strang',
    apply_kwargs=dict(freq_window=None, clamp=np.inf))

fig_sy1 = plot_matrix_field_1d(t_sy, U_sy, xg_sy, component='diag', quantity='real')
display(fig_sy1)

fig_sy2 = plot_matrix_field_1d(t_sy, U_sy, xg_sy, component='frobenius')
display(fig_sy2)

In [ ]:
# Validation against the exact Fourier-space formula
P_lam = sp.lambdify(xi, P_syl, 'numpy')
Q_lam = sp.lambdify(xi, Q_syl, 'numpy')

U0_syl = F_syl(xg_sy)
U0_hat = np.empty((2, 2, len(xg_sy)), dtype=complex)
for i in range(2):
    for j in range(2):
        U0_hat[i, j] = np.fft.fft(U0_syl[i, j])

t_end = t_sy[-1]
U_num_hat = np.fft.fft(U_sy[-1], axis=-1)
err_by_k = np.zeros_like(kx_sy)
for m, k in enumerate(kx_sy):
    El = expm(t_end * np.asarray(P_lam(k), dtype=complex))
    Er = expm(-t_end * np.asarray(Q_lam(k), dtype=complex))
    U_ex_hat = El @ U0_hat[:, :, m] @ Er
    err_by_k[m] = np.max(np.abs(U_num_hat[:, :, m] - U_ex_hat))

k_cut = 20.0
mask = np.abs(kx_sy) <= k_cut
print(f'max Fourier-space error over |k| <= {k_cut}: {err_by_k[mask].max():.3e}')

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.semilogy(kx_sy, err_by_k + 1e-16, '.')
ax.axvline(k_cut, color='r', ls='--', lw=1, label='validation cutoff')
ax.set_xlabel('k'); ax.set_ylabel('max entry error')
ax.set_title('Strang splitting vs exact Sylvester solution')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## Part 8 — `solve_ricci_flow_conformal_2d`: conformal Ricci flow

`g = e^{2φ}(dx² + dy²)` on the flat torus evolves by `∂ₜφ = e^{−2φ}·Δφ`.

Two diagnostics make the example non-trivial:
- **total area** `A = ∫ e^{2φ} dxdy` is conserved (`dA/dt = 2∫Δφ = 0`);
- **Gauss–Bonnet:** `∫ K dA = 0` on the torus, with `K = −e^{−2φ}Δφ` computed here with a `PseudoDifferentialOperator`.

In [ ]:
L_r = 4.0
def phi0_2d(X, Y):
    return (0.25*np.exp(-(X**2 + Y**2))
            + 0.15*np.cos(2*np.pi*X/L_r)*np.cos(2*np.pi*Y/L_r))

t_r, phi_r, (xg_r, yg_r) = solve_ricci_flow_conformal_2d(
    phi0_2d, dt=0.005, n_steps=200, L=L_r, N=64)

n_r = len(t_r)
fig_r = plot_scalar_2d(t_r, phi_r, xg_r, yg_r, quantity='real',
                       times=[0, n_r//2, n_r - 1])
display(fig_r)

In [ ]:
# Invariants: area conservation and Gauss–Bonnet
dxr = xg_r[1] - xg_r[0]; dyr = yg_r[1] - yg_r[0]
area = dxr*dyr*np.sum(np.exp(2*phi_r), axis=(1, 2))

kx_r = 2*np.pi*np.fft.fftfreq(len(xg_r), d=dxr)
ky_r = 2*np.pi*np.fft.fftfreq(len(yg_r), d=dyr)
xs_r, ys_r, xis_r, etas_r = sp.symbols('x y xi eta', real=True)
lap2d = PseudoDifferentialOperator(-(xis_r**2 + etas_r**2), [xs_r, ys_r], mode='symbol')

def curvature(phi):
    lap_phi = lap2d.apply(phi, xg_r, kx_r, y_grid=yg_r, ky=ky_r,
                          freq_window=None, clamp=np.inf).real
    return -np.exp(-2*phi)*lap_phi

K0 = curvature(phi_r[0])
K1 = curvature(phi_r[-1])
intK0 = dxr*dyr*np.sum(K0*np.exp(2*phi_r[0]))
intK1 = dxr*dyr*np.sum(K1*np.exp(2*phi_r[-1]))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
im0 = ax[0].pcolormesh(xg_r, yg_r, K0.T, shading='auto', cmap='RdBu_r')
ax[0].set_title('Gauss curvature K at t = 0'); fig.colorbar(im0, ax=ax[0])
im1 = ax[1].pcolormesh(xg_r, yg_r, K1.T, shading='auto', cmap='RdBu_r')
ax[1].set_title(f'Gauss curvature K at t = {t_r[-1]:.2f}'); fig.colorbar(im1, ax=ax[1])
ax[2].plot(t_r, area/area[0] - 1.0)
ax[2].axhline(0.0, color='k', lw=0.6)
ax[2].set_xlabel('t'); ax[2].set_ylabel('A(t)/A(0) - 1')
ax[2].set_title('Total area conservation'); ax[2].grid(alpha=0.3)
fig.suptitle(f'∫ K dA = {intK0:.2e} (t=0), {intK1:.2e} (final)  —  Gauss–Bonnet: 0', y=1.02)
fig.tight_layout(); plt.show()

## Summary — what was illustrated where

| Function | Example |
|---|---|
| `characteristic_hamiltonians` | harmonic oscillator (1 branch), Dirac (± energy sheets) |
| `integrate_singularity` | circular orbits, Dirac splitting, chaotic Hénon–Heiles + Poincaré section |
| `animate_singularity` / `animate_singularity_3d` | phase-space splitting, 3D chaotic ray tubes |
| `build_propagator` | convergence vs order against `e^{dt·s}`, `expm` check for the Dirac matrix |
| `solve_first_order` | variable-coefficient advection–diffusion, Dirac packet splitting, 2D quantum Hénon–Heiles |
| `solve_second_order` | Klein–Gordon scattering on a barrier (`plot_wave_solution_1d`) |
| `solve_matrix_field` | 1D/2D matrix fields, `plot_matrix_field_1d` / `plot_matrix_field_2d` |
| `solve_sylvester_field` | Dirac ⊕ heat Strang splitting, exact Fourier-space validation |
| `solve_ricci_flow_conformal_2d` | conformal flow with area + Gauss–Bonnet diagnostics |
| `plot_scalar_1d` / `plot_matrix_1d` / `plot_scalar_2d` / `animate_scalar_1d` | scalar & multi-component fields, 2D snapshots, animation |